In [ ]:
from google.colab import files
uploaded = files.upload()


Saving Sentiment analysis.csv to Sentiment analysis.csv
Saving Player_Stats.csv to Player_Stats.csv
Saving Market value data.csv to Market value data.csv
Saving Injury data.csv to Injury data.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

# ==========================
# 1️⃣ Load & Basic Cleaning
# ==========================
market_value_data = pd.read_csv('Market value data.csv', encoding='latin1')
market_value_data.columns = market_value_data.columns.str.strip()
market_value_data.drop_duplicates(inplace=True)

# Separate numeric & categorical columns
num_cols = market_value_data.select_dtypes(include=[np.number]).columns
cat_cols = market_value_data.select_dtypes(include=['object']).columns

# Handle missing values
market_value_data[num_cols] = market_value_data[num_cols].fillna(market_value_data[num_cols].mean())
market_value_data[cat_cols] = market_value_data[cat_cols].fillna(market_value_data[cat_cols].mode().iloc[0])


# ==========================
# 2️⃣ Core Engineered Features
# ==========================
# Removing lines that use columns not present in the dataframe
# market_value_data = market_value_data.assign(
#     Value_Age_Ratio = market_value_data['market_value'] / market_value_data['age'].replace(0, np.nan),
#     Value_Change = market_value_data['highest_value'] - market_value_data['market_value'] # Assuming 'highest_value' exists
# ).copy()


# ==========================
# 3️⃣ Advanced Feature Creation
# ==========================
# Add more domain-specific features based on available columns in Market value data.csv
# For demonstration, let's create a simple interaction and a ratio
if 'international_caps' in market_value_data.columns and 'international_goals' in market_value_data.columns:
    market_value_data['International_Impact'] = market_value_data['international_caps'] * market_value_data['international_goals']

if 'games_played' in market_value_data.columns and 'goals_scored' in market_value_data.columns:
     market_value_data['Goal_Per_Game_Ratio'] = market_value_data['goals_scored'] / market_value_data['games_played'].replace(0, np.nan)


# ==========================
# 4️⃣ Feature Transformation
# ==========================
# Outlier capping (Winsorization)
for col in market_value_data.select_dtypes(include=np.number).columns:
    lower = market_value_data[col].quantile(0.01)
    upper = market_value_data[col].quantile(0.99)
    market_value_data[col] = np.clip(market_value_data[col], lower, upper)


# Impute missing values in numerical features
imputer = SimpleImputer(strategy='mean')
market_value_data[num_cols] = imputer.fit_transform(market_value_data[num_cols])

# Scaling numeric features
scaler = StandardScaler()
num_features_after_impute = market_value_data.select_dtypes(include=np.number).columns
market_value_data[num_features_after_impute] = scaler.fit_transform(market_value_data[num_features_after_impute])


# ==========================
# 5️⃣ Encoding Categorical Features
# ==========================
if len(cat_cols) > 0:
    encoder = OneHotEncoder(sparse_output=False, drop='first')
    encoded = encoder.fit_transform(market_value_data[cat_cols])
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(cat_cols))
    market_value_data.reset_index(drop=True, inplace=True)
    encoded_df.reset_index(drop=True, inplace=True)
    market_value_data = pd.concat([market_value_data, encoded_df], axis=1)
    market_value_data.drop(columns=cat_cols, inplace=True)

# ==========================
# 6️⃣ Polynomial Feature Expansion
# ==========================
# Impute missing values in numerical features before polynomial expansion and PCA
imputer_poly = SimpleImputer(strategy='mean')
numerical_cols_after_encoding = market_value_data.select_dtypes(include=np.number).columns
market_value_data[numerical_cols_after_encoding] = imputer_poly.fit_transform(market_value_data[numerical_cols_after_encoding])


poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly_features = poly.fit_transform(market_value_data[numerical_cols_after_encoding]) # Ensure no NaNs before this step
poly_df = pd.DataFrame(poly_features, columns=poly.get_feature_names_out(numerical_cols_after_encoding))

# Remove duplicated or redundant features
poly_df = poly_df.loc[:, ~poly_df.columns.duplicated()]

# Merge polynomial interactions
market_value_data = pd.concat([market_value_data.reset_index(drop=True), poly_df.reset_index(drop=True)], axis=1)


# ==========================
# 7️⃣ Dimensionality Reduction (PCA)
# ==========================
pca = PCA(n_components=0.95, random_state=42)
pca_features = pca.fit_transform(market_value_data.select_dtypes(include=np.number)) # Ensure no NaNs before this step
pca_df = pd.DataFrame(pca_features, columns=[f'PCA_{i+1}' for i in range(pca_features.shape[1])])

market_value_data = pd.concat([market_value_data.reset_index(drop=True), pca_df.reset_index(drop=True)], axis=1)

# ==========================
# ✅ Final Engineered Dataset
# ==========================
print("✅ Advanced feature engineering complete for Market Value Data!")
print("Shape:", market_value_data.shape)
print(market_value_data.head())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

# ==========================
# 1️⃣ Load & Basic Cleaning
# ==========================
player_stats = pd.read_csv('Player_Stats.csv', encoding='latin1', sep=';')
player_stats.columns = player_stats.columns.str.strip()
player_stats.drop_duplicates(inplace=True)

# Handle missing values (initial handling)
num_cols = player_stats.select_dtypes(include=[np.number]).columns
player_stats[num_cols] = player_stats[num_cols].fillna(player_stats[num_cols].mean())

cat_cols = player_stats.select_dtypes(include=['object']).columns
player_stats[cat_cols] = player_stats[cat_cols].fillna(player_stats[cat_cols].mode().iloc[0])

# ==========================
# 2️⃣ Core Engineered Features
# ==========================
player_stats = player_stats.assign(
    Performance_Index=(
        (player_stats["Goals"].astype(float) + player_stats["Assists"].astype(float)) /
        player_stats["90s"].replace(0, np.nan)
    ),
    Shot_Accuracy=(
        player_stats["SoT"].astype(float) / player_stats["Shots"].replace(0, np.nan)
    ),
    Pass_Efficiency=(
        player_stats["PasTotCmp"].astype(float) / player_stats["PasTotAtt"].replace(0, np.nan)
    )
).copy()

# ==========================
# 3️⃣ Advanced Feature Creation
# ==========================
# a) Ratios and efficiency metrics
player_stats["Goal_Contribution"] = player_stats["Goals"] + player_stats["Assists"]
player_stats["Goals_per_Shot"] = player_stats["Goals"] / player_stats["Shots"].replace(0, np.nan)
player_stats["SoT_per_90"] = player_stats["SoT"] / player_stats["90s"].replace(0, np.nan)
player_stats["Passing_Impact"] = player_stats["Pass_Efficiency"] * player_stats["Performance_Index"]

# b) Interaction-based features
player_stats["Attack_Effectiveness"] = player_stats["Performance_Index"] * player_stats["Shot_Accuracy"]
player_stats["Overall_Contribution"] = (
    player_stats["Attack_Effectiveness"] +
    player_stats["Passing_Impact"]
)

# c) Normalized ratios (scaled per 90 minutes)
per90_cols = ["Goals", "Assists", "Shots", "SoT", "PasTotAtt", "PasTotCmp"]
for col in per90_cols:
    player_stats[f"{col}_per90"] = player_stats[col] / player_stats["90s"].replace(0, np.nan)

# d) Cumulative performance index (log transformation to reduce skew)
player_stats["Log_Performance_Index"] = np.log1p(player_stats["Performance_Index"])

# ==========================
# 4️⃣ Feature Transformation
# ==========================
# Outlier capping (winsorization)
for col in player_stats.select_dtypes(include=np.number).columns:
    lower = player_stats[col].quantile(0.01)
    upper = player_stats[col].quantile(0.99)
    player_stats[col] = np.clip(player_stats[col], lower, upper)

# Scaling numerical features
scaler = StandardScaler()
num_features = player_stats.select_dtypes(include=np.number).columns
player_stats[num_features] = scaler.fit_transform(player_stats[num_features])

# Impute missing values in numerical features
imputer = SimpleImputer(strategy='mean')
player_stats[num_features] = imputer.fit_transform(player_stats[num_features])

# ==========================
# 5️⃣ Encoding Categorical Features
# ==========================
if len(cat_cols) > 0:
    encoder = OneHotEncoder(sparse_output=False, drop='first')
    encoded = encoder.fit_transform(player_stats[cat_cols])
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(cat_cols))
    player_stats.reset_index(drop=True, inplace=True)
    encoded_df.reset_index(drop=True, inplace=True)
    player_stats = pd.concat([player_stats, encoded_df], axis=1)
    player_stats.drop(columns=cat_cols, inplace=True)

# ==========================
# 6️⃣ Polynomial Feature Expansion
# ==========================
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly_features = poly.fit_transform(player_stats[num_features])
poly_df = pd.DataFrame(poly_features, columns=poly.get_feature_names_out(num_features))

# Drop extremely correlated or redundant polynomial features (optional)
poly_df = poly_df.loc[:, ~poly_df.columns.duplicated()]

# Merge polynomial interactions
player_stats = pd.concat([player_stats.reset_index(drop=True), poly_df.reset_index(drop=True)], axis=1)

# ==========================
# 7️⃣ Dimensionality Reduction (PCA)
# ==========================
pca = PCA(n_components=0.95, random_state=42)
pca_features = pca.fit_transform(player_stats.select_dtypes(include=np.number))
pca_df = pd.DataFrame(pca_features, columns=[f'PCA_{i+1}' for i in range(pca_features.shape[1])])

player_stats = pd.concat([player_stats.reset_index(drop=True), pca_df.reset_index(drop=True)], axis=1)

# ==========================
# ✅ Final Engineered Dataset
# ==========================
print("✅ Advanced feature engineering complete!")
print("Shape:", player_stats.shape)
print(player_stats.head())

✅ Advanced feature engineering complete!
Shape: (2689, 12707)
         Rk       Age      Born        MP    Starts       Min       90s  \
0 -1.697821 -0.912889  0.870634  1.192631  1.506411  1.415170  1.409964   
1 -1.697821  2.046637 -2.087355  1.484467  1.935727  2.065415  2.065287   
2 -1.697821 -0.685233  0.643096  0.317123 -0.067748  0.016467  0.023118   
3 -1.697821 -0.912889  0.870634  1.192631  1.649517  1.758919  1.760486   
4 -1.697821  0.908358 -0.722129  0.463041  0.933990  0.685339  0.678441   

      Goals     Shots       SoT  ...   PCA_641   PCA_642   PCA_643   PCA_644  \
0  0.004272  0.247714 -0.198488  ... -0.685702 -2.966910  1.285750  2.132070   
1 -0.549012 -0.282000 -0.655317  ...  0.235072 -0.439251  1.266417 -0.236572   
2 -0.549012 -0.131782 -0.059453  ... -0.854393 -0.028622  0.423889 -0.060192   
3  0.004272 -0.487561 -0.456696  ... -0.759769  1.276135 -0.303506 -0.261118   
4 -0.549012 -0.716840 -0.754627  ... -0.151745 -0.188199 -0.503701 -0.007309   

    PC

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

# ==========================
# 1️⃣ Load & Basic Cleaning
# ==========================
injury_data = pd.read_csv('Injury data.csv', encoding='latin1')
injury_data.columns = injury_data.columns.str.strip()
injury_data.drop_duplicates(inplace=True)

# Separate numeric & categorical columns
num_cols = injury_data.select_dtypes(include=[np.number]).columns
cat_cols = injury_data.select_dtypes(include=['object']).columns

# Handle missing values
injury_data[num_cols] = injury_data[num_cols].fillna(injury_data[num_cols].mean())
injury_data[cat_cols] = injury_data[cat_cols].fillna(injury_data[cat_cols].mode().iloc[0])

# ==========================
# 2️⃣ Core Engineered Features
# ==========================
injury_data = injury_data.assign(
    Injury_Risk_Score=np.log1p(injury_data["season_days_injured"]),
    Games_Missed_Ratio=1 - (
        injury_data["season_games_played"] /
        injury_data["season_matches_in_squad"].replace(0, np.nan)
    ),
    Injury_Severity_Index=(
        injury_data["total_days_injured"] /
        injury_data["total_games_played"].replace(0, np.nan)
    )
).copy()

# ==========================
# 3️⃣ Advanced Feature Creation
# ==========================
# a) Ratios and per-season normalization
# Removing lines that use columns not present in the dataframe
# injury_data["Avg_Days_Injured_per_Season"] = (
#     injury_data["total_days_injured"] /
#     injury_data["seasons_at_club"].replace(0, np.nan)
# )
# injury_data["Avg_Injuries_per_Season"] = (
#     injury_data["total_injuries"] /
#     injury_data["seasons_at_club"].replace(0, np.nan)
# )
# injury_data["Injury_to_Game_Ratio"] = (
#     injury_data["total_injuries"] /
#     injury_data["total_games_played"].replace(0, np.nan)
# )


# b) Interaction-based features
injury_data["Recovery_Efficiency"] = (
    1 / (1 + injury_data["Injury_Risk_Score"] * injury_data["Injury_Severity_Index"])
)
injury_data["Impact_on_Availability"] = (
    injury_data["Games_Missed_Ratio"] * injury_data["Injury_Severity_Index"]
)

# c) Cumulative log transformation to handle skew
injury_data["Log_Injury_Severity"] = np.log1p(injury_data["Injury_Severity_Index"])
# Removing line that uses 'total_injuries'
# injury_data["Log_Injury_to_Game"] = np.log1p(injury_data["Injury_to_Game_Ratio"])


# ==========================
# 4️⃣ Feature Transformation
# ==========================
# Outlier capping (Winsorization)
for col in injury_data.select_dtypes(include=np.number).columns:
    lower = injury_data[col].quantile(0.01)
    upper = injury_data[col].quantile(0.99)
    injury_data[col] = np.clip(injury_data[col], lower, upper)

# Scaling numeric features
scaler = StandardScaler()
num_features = injury_data.select_dtypes(include=np.number).columns
injury_data[num_features] = scaler.fit_transform(injury_data[num_features])

# ==========================
# 5️⃣ Encoding Categorical Features
# ==========================
if len(cat_cols) > 0:
    encoder = OneHotEncoder(sparse_output=False, drop='first')
    encoded = encoder.fit_transform(injury_data[cat_cols])
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(cat_cols))
    injury_data.reset_index(drop=True, inplace=True)
    encoded_df.reset_index(drop=True, inplace=True)
    injury_data = pd.concat([injury_data, encoded_df], axis=1)
    injury_data.drop(columns=cat_cols, inplace=True)

# ==========================
# 6️⃣ Polynomial Feature Expansion
# ==========================
# Impute missing values in numerical features before polynomial expansion and PCA
imputer = SimpleImputer(strategy='mean')
numerical_cols_after_encoding = injury_data.select_dtypes(include=np.number).columns
injury_data[numerical_cols_after_encoding] = imputer.fit_transform(injury_data[numerical_cols_after_encoding])

poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly_features = poly.fit_transform(injury_data[numerical_cols_after_encoding]) # Ensure no NaNs before this step
poly_df = pd.DataFrame(poly_features, columns=poly.get_feature_names_out(numerical_cols_after_encoding))

# Remove duplicated or redundant features
poly_df = poly_df.loc[:, ~poly_df.columns.duplicated()]

# Merge polynomial interactions
injury_data = pd.concat([injury_data.reset_index(drop=True), poly_df.reset_index(drop=True)], axis=1)

# ==========================
# 7️⃣ Dimensionality Reduction (PCA)
# ==========================
pca = PCA(n_components=0.95, random_state=42)
pca_features = pca.fit_transform(injury_data.select_dtypes(include=np.number)) # Ensure no NaNs before this step
pca_df = pd.DataFrame(pca_features, columns=[f'PCA_{i+1}' for i in range(pca_features.shape[1])])

player_stats = pd.concat([player_stats.reset_index(drop=True), pca_df.reset_index(drop=True)], axis=1)

# ==========================
# ✅ Final Engineered Dataset
# ==========================
print("✅ Advanced feature engineering complete for Injury Data!")
print("Shape:", injury_data.shape)
print(injury_data.head())

# Task
Load the data from "Sentiment analysis.csv", perform advanced feature engineering, and display the head and shape of the resulting DataFrame.

## Load and clean data

### Subtask:
Load the `Sentiment analysis.csv` data into a pandas DataFrame, perform basic cleaning such as handling missing values and duplicates.


**Reasoning**:
Load the sentiment data, perform basic cleaning steps including handling column names, duplicates, and missing values based on column types.



In [1]:
sentiment_data = pd.read_csv('Sentiment analysis.csv', encoding='latin1')
sentiment_data.columns = sentiment_data.columns.str.strip()
sentiment_data.drop_duplicates(inplace=True)

num_cols = sentiment_data.select_dtypes(include=[np.number]).columns
cat_cols = sentiment_data.select_dtypes(include=['object']).columns

sentiment_data[num_cols] = sentiment_data[num_cols].fillna(sentiment_data[num_cols].mean())
sentiment_data[cat_cols] = sentiment_data[cat_cols].fillna(sentiment_data[cat_cols].mode().iloc[0])

NameError: name 'pd' is not defined

**Reasoning**:
The previous command failed because pandas was not imported. I need to import pandas before using it to load and process the data.



In [2]:
import pandas as pd
import numpy as np

sentiment_data = pd.read_csv('Sentiment analysis.csv', encoding='latin1')
sentiment_data.columns = sentiment_data.columns.str.strip()
sentiment_data.drop_duplicates(inplace=True)

num_cols = sentiment_data.select_dtypes(include=[np.number]).columns
cat_cols = sentiment_data.select_dtypes(include=['object']).columns

sentiment_data[num_cols] = sentiment_data[num_cols].fillna(sentiment_data[num_cols].mean())
sentiment_data[cat_cols] = sentiment_data[cat_cols].fillna(sentiment_data[cat_cols].mode().iloc[0])

FileNotFoundError: [Errno 2] No such file or directory: 'Sentiment analysis.csv'

**Reasoning**:
The previous command failed because the file was not found. I need to ensure the file exists before attempting to load it. Based on the notebook history, the file was uploaded in the first cell. I will re-run the code to load and clean the data, including the necessary imports.



In [3]:
import pandas as pd
import numpy as np

sentiment_data = pd.read_csv('Sentiment analysis.csv', encoding='latin1')
sentiment_data.columns = sentiment_data.columns.str.strip()
sentiment_data.drop_duplicates(inplace=True)

num_cols = sentiment_data.select_dtypes(include=[np.number]).columns
cat_cols = sentiment_data.select_dtypes(include=['object']).columns

sentiment_data[num_cols] = sentiment_data[num_cols].fillna(sentiment_data[num_cols].mean())
sentiment_data[cat_cols] = sentiment_data[cat_cols].fillna(sentiment_data[cat_cols].mode().iloc[0])

FileNotFoundError: [Errno 2] No such file or directory: 'Sentiment analysis.csv'

## Load and clean data

### Subtask:
Load the `Sentiment analysis.csv` data into a pandas DataFrame, perform basic cleaning such as handling missing values and duplicates.


**Reasoning**:
Load the sentiment data and perform basic cleaning steps including handling column names, duplicates, and missing values.



In [4]:
sentiment_data = pd.read_csv('Sentiment analysis.csv', encoding='latin1')
sentiment_data.columns = sentiment_data.columns.str.strip()
sentiment_data.drop_duplicates(inplace=True)

num_cols = sentiment_data.select_dtypes(include=[np.number]).columns
cat_cols = sentiment_data.select_dtypes(include=['object']).columns

sentiment_data[num_cols] = sentiment_data[num_cols].fillna(sentiment_data[num_cols].mean())
sentiment_data[cat_cols] = sentiment_data[cat_cols].fillna(sentiment_data[cat_cols].mode().iloc[0])

FileNotFoundError: [Errno 2] No such file or directory: 'Sentiment analysis.csv'

## Feature engineering

### Subtask:
Extract relevant features from the text data, such as sentiment scores, emotional tone, or topic modeling features.


**Reasoning**:
Analyze the columns to identify the text column and then apply sentiment analysis to extract sentiment scores.



In [5]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

# Check if 'nltk_data' is available, if not download it
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except nltk.downloader.DownloadError:
    nltk.download('vader_lexicon')
except LookupError:
    nltk.download('vader_lexicon')


# Assuming the text column is named 'text' or 'review' or similar.
# Replace 'text_column_name' with the actual name of the text column in your DataFrame.
# You may need to inspect the DataFrame columns first if you are unsure.
text_column_name = None
for col in sentiment_data.columns:
    if sentiment_data[col].dtype == 'object' and sentiment_data[col].str.len().mean() > 10: # Heuristic to find text column
        text_column_name = col
        break

if text_column_name:
    sid = SentimentIntensityAnalyzer()
    sentiment_data['sentiment_score'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(text)['compound'])
    sentiment_data['sentiment_positive'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(text)['pos'])
    sentiment_data['sentiment_negative'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(text)['neg'])
    sentiment_data['sentiment_neutral'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(text)['neu'])
else:
    print("Could not identify the text column in the DataFrame.")

print("Shape after sentiment analysis:", sentiment_data.shape)
print(sentiment_data.head())

AttributeError: module 'nltk.downloader' has no attribute 'DownloadError'

**Reasoning**:
The previous attempt to download the VADER lexicon failed due to an AttributeError. I will correct the exception handling for downloading the lexicon and re-run the sentiment analysis.



In [6]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

# Corrected exception handling for downloading the lexicon
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon')

# Assuming the text column is named 'text' or 'review' or similar.
# Replace 'text_column_name' with the actual name of the text column in your DataFrame.
# You may need to inspect the DataFrame columns first if you are unsure.
text_column_name = None
for col in sentiment_data.columns:
    if sentiment_data[col].dtype == 'object' and sentiment_data[col].str.len().mean() > 10: # Heuristic to find text column
        text_column_name = col
        break

if text_column_name:
    sid = SentimentIntensityAnalyzer()
    sentiment_data['sentiment_score'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(text)['compound'])
    sentiment_data['sentiment_positive'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(text)['pos'])
    sentiment_data['sentiment_negative'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(text)['neg'])
    sentiment_data['sentiment_neutral'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(text)['neu'])
else:
    print("Could not identify the text column in the DataFrame.")

print("Shape after sentiment analysis:", sentiment_data.shape)
print(sentiment_data.head())

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


NameError: name 'sentiment_data' is not defined

**Reasoning**:
The error indicates that the `sentiment_data` DataFrame is not defined. This is because the previous attempts to load the data failed. I need to re-load the data before performing sentiment analysis.



In [7]:
import pandas as pd
import numpy as np
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

# Check if 'nltk_data' is available, if not download it
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon')

# Load the data again as the DataFrame was not defined in the previous attempt
try:
    sentiment_data = pd.read_csv('Sentiment analysis.csv', encoding='latin1')
    sentiment_data.columns = sentiment_data.columns.str.strip()
    sentiment_data.drop_duplicates(inplace=True)

    num_cols = sentiment_data.select_dtypes(include=[np.number]).columns
    cat_cols = sentiment_data.select_dtypes(include=['object']).columns

    sentiment_data[num_cols] = sentiment_data[num_cols].fillna(sentiment_data[num_cols].mean())
    sentiment_data[cat_cols] = sentiment_data[cat_cols].fillna(sentiment_data[cat_cols].mode().iloc[0])

    # Assuming the text column is named 'text' or 'review' or similar.
    # Replace 'text_column_name' with the actual name of the text column in your DataFrame.
    # You may need to inspect the DataFrame columns first if you are unsure.
    text_column_name = None
    for col in sentiment_data.columns:
        if sentiment_data[col].dtype == 'object' and sentiment_data[col].str.len().mean() > 10: # Heuristic to find text column
            text_column_name = col
            break

    if text_column_name:
        sid = SentimentIntensityAnalyzer()
        sentiment_data['sentiment_score'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(text)['compound'])
        sentiment_data['sentiment_positive'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(text)['pos'])
        sentiment_data['sentiment_negative'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(text)['neg'])
        sentiment_data['sentiment_neutral'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(text)['neu'])

        print("Shape after sentiment analysis:", sentiment_data.shape)
        print(sentiment_data.head())
    else:
        print("Could not identify the text column in the DataFrame.")

except FileNotFoundError:
    print("Error: 'Sentiment analysis.csv' not found. Please ensure the file is uploaded.")
except Exception as e:
    print(f"An error occurred: {e}")


Error: 'Sentiment analysis.csv' not found. Please ensure the file is uploaded.


## Feature transformation

### Subtask:
Apply transformations to the engineered features, such as scaling or normalization.


**Reasoning**:
Load the data again as the DataFrame was not successfully loaded in previous attempts and then apply scaling to the numerical columns.



In [8]:
try:
    sentiment_data = pd.read_csv('Sentiment analysis.csv', encoding='latin1')
    sentiment_data.columns = sentiment_data.columns.str.strip()
    sentiment_data.drop_duplicates(inplace=True)

    # Handle missing values (basic cleaning re-applied)
    num_cols = sentiment_data.select_dtypes(include=[np.number]).columns
    cat_cols = sentiment_data.select_dtypes(include=['object']).columns

    sentiment_data[num_cols] = sentiment_data[num_cols].fillna(sentiment_data[num_cols].mean())
    sentiment_data[cat_cols] = sentiment_data[cat_cols].fillna(sentiment_data[cat_cols].mode().iloc[0])

    # Apply scaling to numerical features
    scaler = StandardScaler()
    sentiment_data[num_cols] = scaler.fit_transform(sentiment_data[num_cols])

    print("Numerical columns scaled.")
    print("Shape:", sentiment_data.shape)
    display(sentiment_data.head())

except FileNotFoundError:
    print("Error: 'Sentiment analysis.csv' not found. Please ensure the file is uploaded.")
except Exception as e:
    print(f"An error occurred: {e}")

Error: 'Sentiment analysis.csv' not found. Please ensure the file is uploaded.


## Encoding categorical features

### Subtask:
Encode any categorical features in the dataset.


**Reasoning**:
Load the data, perform basic cleaning, and then identify and encode the categorical features using one-hot encoding as instructed.



In [9]:
from sklearn.preprocessing import OneHotEncoder

# Load the data again as the DataFrame was not defined in the previous attempt
try:
    sentiment_data = pd.read_csv('Sentiment analysis.csv', encoding='latin1')
    sentiment_data.columns = sentiment_data.columns.str.strip()
    sentiment_data.drop_duplicates(inplace=True)

    # Handle missing values (basic cleaning re-applied)
    num_cols = sentiment_data.select_dtypes(include=[np.number]).columns
    cat_cols = sentiment_data.select_dtypes(include=['object']).columns

    sentiment_data[num_cols] = sentiment_data[num_cols].fillna(sentiment_data[num_cols].mean())
    sentiment_data[cat_cols] = sentiment_data[cat_cols].fillna(sentiment_data[cat_cols].mode().iloc[0])

    # Identify categorical columns again after cleaning
    cat_cols = sentiment_data.select_dtypes(include=['object']).columns

    # Encode categorical features if they exist
    if len(cat_cols) > 0:
        encoder = OneHotEncoder(sparse_output=False, drop='first')
        encoded = encoder.fit_transform(sentiment_data[cat_cols])
        encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(cat_cols))

        # Concatenate original DataFrame (excluding original categorical columns) with encoded DataFrame
        sentiment_data.reset_index(drop=True, inplace=True)
        encoded_df.reset_index(drop=True, inplace=True)
        sentiment_data = pd.concat([sentiment_data.drop(columns=cat_cols), encoded_df], axis=1)

        print("Categorical features encoded.")

    print("Shape after encoding:", sentiment_data.shape)
    display(sentiment_data.head())

except FileNotFoundError:
    print("Error: 'Sentiment analysis.csv' not found. Please ensure the file is uploaded.")
except Exception as e:
    print(f"An error occurred: {e}")

Error: 'Sentiment analysis.csv' not found. Please ensure the file is uploaded.


## Dimensionality reduction

### Subtask:
Reduce the dimensionality of the feature space using techniques like PCA if necessary.


**Reasoning**:
I need to load the data again to ensure the DataFrame is available for the PCA step, as previous attempts failed due to FileNotFoundError. After loading and basic cleaning, I will select the numerical columns, apply PCA to reduce dimensionality, and then combine the PCA results with the non-numerical columns.



In [10]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

# Load the data again as the DataFrame was not defined in the previous attempt
try:
    sentiment_data = pd.read_csv('Sentiment analysis.csv', encoding='latin1')
    sentiment_data.columns = sentiment_data.columns.str.strip()
    sentiment_data.drop_duplicates(inplace=True)

    # Handle missing values (basic cleaning re-applied)
    num_cols = sentiment_data.select_dtypes(include=[np.number]).columns
    cat_cols = sentiment_data.select_dtypes(include=['object']).columns

    sentiment_data[num_cols] = sentiment_data[num_cols].fillna(sentiment_data[num_cols].mean())
    sentiment_data[cat_cols] = sentiment_data[cat_cols].fillna(sentiment_data[cat_cols].mode().iloc[0])

    # Select numerical columns for PCA
    numerical_cols = sentiment_data.select_dtypes(include=np.number).columns

    # Impute missing values in numerical features before PCA
    imputer = SimpleImputer(strategy='mean')
    sentiment_data[numerical_cols] = imputer.fit_transform(sentiment_data[numerical_cols])

    # Scale numerical features before PCA
    scaler = StandardScaler()
    scaled_numerical_data = scaler.fit_transform(sentiment_data[numerical_cols])
    scaled_numerical_df = pd.DataFrame(scaled_numerical_data, columns=numerical_cols)


    # Apply PCA
    pca = PCA(n_components=0.95, random_state=42)
    pca_features = pca.fit_transform(scaled_numerical_df)

    # Create a new DataFrame from PCA results
    pca_df = pd.DataFrame(pca_features, columns=[f'PCA_{i+1}' for i in range(pca_features.shape[1])])

    # Separate non-numerical columns
    non_numerical_cols = sentiment_data.select_dtypes(exclude=np.number).columns
    non_numerical_df = sentiment_data[non_numerical_cols].reset_index(drop=True)

    # Concatenate non-numerical columns with PCA DataFrame
    sentiment_data_pca = pd.concat([non_numerical_df, pca_df.reset_index(drop=True)], axis=1)

    print("Dimensionality reduction with PCA complete.")
    print("Shape after PCA:", sentiment_data_pca.shape)
    display(sentiment_data_pca.head())

except FileNotFoundError:
    print("Error: 'Sentiment analysis.csv' not found. Please ensure the file is uploaded.")
except Exception as e:
    print(f"An error occurred: {e}")

Error: 'Sentiment analysis.csv' not found. Please ensure the file is uploaded.


## Combine features

### Subtask:
Combine all engineered and transformed features into a final dataset ready for modeling.


**Reasoning**:
The previous attempts failed because the data file was not found. I will try loading the data again and then proceed with the steps outlined in the subtask: identify text column, perform sentiment analysis, identify numerical columns, impute missing values, scale numerical features, identify categorical columns, encode categorical features, select numerical columns again, impute missing values, apply PCA, separate non-numerical columns, and concatenate the results.



In [11]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

# Check if 'nltk_data' is available, if not download it
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon')

# Load the data
try:
    sentiment_data = pd.read_csv('Sentiment analysis.csv', encoding='latin1')
    sentiment_data.columns = sentiment_data.columns.str.strip()
    sentiment_data.drop_duplicates(inplace=True)

    # Handle missing values (basic cleaning)
    num_cols_initial = sentiment_data.select_dtypes(include=[np.number]).columns
    cat_cols_initial = sentiment_data.select_dtypes(include=['object']).columns

    sentiment_data[num_cols_initial] = sentiment_data[num_cols_initial].fillna(sentiment_data[num_cols_initial].mean())
    sentiment_data[cat_cols_initial] = sentiment_data[cat_cols_initial].fillna(sentiment_data[cat_cols_initial].mode().iloc[0])

    # Identify the text column
    text_column_name = None
    for col in sentiment_data.columns:
        if sentiment_data[col].dtype == 'object' and sentiment_data[col].str.len().mean() > 10:
            text_column_name = col
            break

    # Perform sentiment analysis if text column is found
    if text_column_name:
        sid = SentimentIntensityAnalyzer()
        sentiment_data['sentiment_score'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(str(text))['compound'])
        sentiment_data['sentiment_positive'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(str(text))['pos'])
        sentiment_data['sentiment_negative'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(str(text))['neg'])
        sentiment_data['sentiment_neutral'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(str(text))['neu'])
        print("Sentiment analysis complete.")
    else:
        print("Could not identify the text column for sentiment analysis.")

    # Identify numerical columns after adding sentiment scores
    numerical_cols_after_sentiment = sentiment_data.select_dtypes(include=np.number).columns

    # Impute missing values in numerical features
    imputer = SimpleImputer(strategy='mean')
    sentiment_data[numerical_cols_after_sentiment] = imputer.fit_transform(sentiment_data[numerical_cols_after_sentiment])
    print("Numerical imputation complete.")


    # Scale numerical features
    scaler = StandardScaler()
    sentiment_data[numerical_cols_after_sentiment] = scaler.fit_transform(sentiment_data[numerical_cols_after_sentiment])
    print("Numerical scaling complete.")

    # Identify categorical columns
    cat_cols_after_sentiment = sentiment_data.select_dtypes(include=['object']).columns

    # Encode categorical features if they exist
    if len(cat_cols_after_sentiment) > 0:
        encoder = OneHotEncoder(sparse_output=False, drop='first')
        encoded = encoder.fit_transform(sentiment_data[cat_cols_after_sentiment])
        encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(cat_cols_after_sentiment))

        # Concatenate original DataFrame (excluding original categorical columns) with encoded DataFrame
        sentiment_data.reset_index(drop=True, inplace=True)
        encoded_df.reset_index(drop=True, inplace=True)
        sentiment_data = pd.concat([sentiment_data.drop(columns=cat_cols_after_sentiment), encoded_df], axis=1)

        print("Categorical features encoded.")

    # Select numerical columns again after encoding and impute if any new NaNs were introduced (unlikely but good practice)
    numerical_cols_after_encoding = sentiment_data.select_dtypes(include=np.number).columns
    imputer_final = SimpleImputer(strategy='mean')
    sentiment_data[numerical_cols_after_encoding] = imputer_final.fit_transform(sentiment_data[numerical_cols_after_encoding])
    print("Final numerical imputation complete.")


    # Apply PCA
    pca = PCA(n_components=0.95, random_state=42)
    pca_features = pca.fit_transform(sentiment_data[numerical_cols_after_encoding])
    pca_df = pd.DataFrame(pca_features, columns=[f'PCA_{i+1}' for i in range(pca_features.shape[1])])
    print(f"PCA complete, retained {pca_features.shape[1]} components.")


    # Separate non-numerical columns
    non_numerical_cols_final = sentiment_data.select_dtypes(exclude=np.number).columns
    non_numerical_df = sentiment_data[non_numerical_cols_final].reset_index(drop=True)

    # Concatenate non-numerical columns with PCA DataFrame
    sentiment_data_final = pd.concat([non_numerical_df, pca_df.reset_index(drop=True)], axis=1)

    print("\n✅ Advanced feature engineering complete for Sentiment Analysis Data!")
    print("Shape:", sentiment_data_final.shape)
    display(sentiment_data_final.head())

except FileNotFoundError:
    print("Error: 'Sentiment analysis.csv' not found. Please ensure the file is uploaded.")
except Exception as e:
    print(f"An error occurred: {e}")

Error: 'Sentiment analysis.csv' not found. Please ensure the file is uploaded.


In [13]:
from google.colab import files
uploaded = files.upload()

Saving Sentiment analysis.csv to Sentiment analysis.csv


**Reasoning**:
The previous attempt to load the data failed because the file was not found. I will try loading the data again and if successful, I will continue with the remaining steps of the subtask: perform sentiment analysis, handle missing values, scale numerical features, encode categorical features, select numerical columns again, impute missing values, apply PCA, separate non-numerical columns, and concatenate the results.



In [14]:
# Load the data again
try:
    sentiment_data = pd.read_csv('Sentiment analysis.csv', encoding='latin1')
    sentiment_data.columns = sentiment_data.columns.str.strip()
    sentiment_data.drop_duplicates(inplace=True)

    # Handle missing values (basic cleaning)
    num_cols_initial = sentiment_data.select_dtypes(include=[np.number]).columns
    cat_cols_initial = sentiment_data.select_dtypes(include=['object']).columns

    sentiment_data[num_cols_initial] = sentiment_data[num_cols_initial].fillna(sentiment_data[num_cols_initial].mean())
    sentiment_data[cat_cols_initial] = sentiment_data[cat_cols_initial].fillna(sentiment_data[cat_cols_initial].mode().iloc[0])

    # Identify the text column
    text_column_name = None
    for col in sentiment_data.columns:
        if sentiment_data[col].dtype == 'object' and sentiment_data[col].str.len().mean() > 10:
            text_column_name = col
            break

    # Perform sentiment analysis if text column is found
    if text_column_name:
        sid = SentimentIntensityAnalyzer()
        sentiment_data['sentiment_score'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(str(text))['compound'])
        sentiment_data['sentiment_positive'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(str(text))['pos'])
        sentiment_data['sentiment_negative'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(str(text))['neg'])
        sentiment_data['sentiment_neutral'] = sentiment_data[text_column_name].apply(lambda text: sid.polarity_scores(str(text))['neu'])
        print("Sentiment analysis complete.")
    else:
        print("Could not identify the text column for sentiment analysis.")

    # Identify numerical columns after adding sentiment scores
    numerical_cols_after_sentiment = sentiment_data.select_dtypes(include=np.number).columns

    # Impute missing values in numerical features
    imputer = SimpleImputer(strategy='mean')
    sentiment_data[numerical_cols_after_sentiment] = imputer.fit_transform(sentiment_data[numerical_cols_after_sentiment])
    print("Numerical imputation complete.")


    # Scale numerical features
    scaler = StandardScaler()
    sentiment_data[numerical_cols_after_sentiment] = scaler.fit_transform(sentiment_data[numerical_cols_after_sentiment])
    print("Numerical scaling complete.")

    # Identify categorical columns
    cat_cols_after_sentiment = sentiment_data.select_dtypes(include=['object']).columns

    # Encode categorical features if they exist
    if len(cat_cols_after_sentiment) > 0:
        encoder = OneHotEncoder(sparse_output=False, drop='first')
        encoded = encoder.fit_transform(sentiment_data[cat_cols_after_sentiment])
        encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(cat_cols_after_sentiment))

        # Concatenate original DataFrame (excluding original categorical columns) with encoded DataFrame
        sentiment_data.reset_index(drop=True, inplace=True)
        encoded_df.reset_index(drop=True, inplace=True)
        sentiment_data = pd.concat([sentiment_data.drop(columns=cat_cols_after_sentiment), encoded_df], axis=1)

        print("Categorical features encoded.")

    # Select numerical columns again after encoding and impute if any new NaNs were introduced (unlikely but good practice)
    numerical_cols_after_encoding = sentiment_data.select_dtypes(include=np.number).columns
    imputer_final = SimpleImputer(strategy='mean')
    sentiment_data[numerical_cols_after_encoding] = imputer_final.fit_transform(sentiment_data[numerical_cols_after_encoding])
    print("Final numerical imputation complete.")


    # Apply PCA
    pca = PCA(n_components=0.95, random_state=42)
    pca_features = pca.fit_transform(sentiment_data[numerical_cols_after_encoding])
    pca_df = pd.DataFrame(pca_features, columns=[f'PCA_{i+1}' for i in range(pca_features.shape[1])])
    print(f"PCA complete, retained {pca_features.shape[1]} components.")


    # Separate non-numerical columns
    non_numerical_cols_final = sentiment_data.select_dtypes(exclude=np.number).columns
    non_numerical_df = sentiment_data[non_numerical_cols_final].reset_index(drop=True)

    # Concatenate non-numerical columns with PCA DataFrame
    sentiment_data_final = pd.concat([non_numerical_df, pca_df.reset_index(drop=True)], axis=1)

    print("\n✅ Advanced feature engineering complete for Sentiment Analysis Data!")
    print("Shape:", sentiment_data_final.shape)
    display(sentiment_data_final.head())

except FileNotFoundError:
    print("Error: 'Sentiment analysis.csv' not found. Please ensure the file is uploaded.")
except Exception as e:
    print(f"An error occurred: {e}")

Sentiment analysis complete.
Numerical imputation complete.
Numerical scaling complete.
Categorical features encoded.
Final numerical imputation complete.
PCA complete, retained 530 components.

✅ Advanced feature engineering complete for Sentiment Analysis Data!
Shape: (732, 530)


,PCA_1,PCA_2,PCA_3,PCA_4,PCA_5,PCA_6,PCA_7,PCA_8,PCA_9,PCA_10,...,PCA_521,PCA_522,PCA_523,PCA_524,PCA_525,PCA_526,PCA_527,PCA_528,PCA_529,PCA_530
0,-2.254768,3.714775,0.016402,1.139666,0.041912,-0.049777,0.488062,0.020810,-0.524665,-0.469811,...,-1.257980e-16,1.607875e-16,-6.059342e-17,9.989113e-17,-3.284432e-17,-1.375129e-16,-3.259658e-16,-4.324741e-18,-9.374688e-17,-1.277077e-16
1,-5.462819,-0.609918,0.272414,1.100188,0.252722,-0.665675,-0.370783,-0.327321,-0.324118,-0.390750,...,3.489079e-02,-2.654198e-02,1.163253e-02,3.309795e-02,-8.074470e-03,-6.774483e-03,-8.273826e-03,-1.844174e-02,-3.708516e-03,-1.625927e-02
2,-1.506979,2.613820,1.198970,0.439817,0.185022,-0.131354,0.891697,0.509836,0.561915,0.279272,...,1.901463e-02,-2.415168e-02,-1.333906e-03,1.454448e-02,-4.258227e-03,1.364585e-02,-2.343242e-02,1.682441e-02,4.241752e-03,-9.857539e-03
3,-3.031256,1.597932,0.905753,0.858525,-1.155080,1.130892,1.487099,0.073543,-0.177131,0.139360,...,-3.734483e-03,-1.315475e-02,2.137218e-02,1.367081e-02,6.509157e-03,-6.534142e-02,3.536280e-02,-3.382303e-02,-1.882534e-02,-9.938898e-03
4,-2.605264,-0.235433,2.667963,-0.053832,-1.382095,0.743067,1.353050,0.444030,0.654178,0.240761,...,3.340158e-02,-1.572683e-03,1.245022e-02,2.629060e-02,1.033976e-02,2.576957e-02,-1.233401e-02,-2.277084e-02,-5.302959e-03,-3.598550e-02
